# MSC Supplementary Figure S7 — Final Sensitivity Analysis

This notebook uses the frozen `msc_model_feedback.py` simulator and the exact intervals in `config_msc_feedback.yaml`. It generates the final global sensitivity analysis, targeted profiles of the principal phenomenological parameters, and Hill-coefficient sensitivity of history dependence.

**Design**
- 500 Latin-hypercube parameter sets across all 27 sampled parameters.
- Four independent stochastic replicates per parameter set.
- Thirty replicates per targeted profile point.
- Fifteen hysteresis replicates for each \(n_H\in\{3,4,5,6\}\).
- Spearman correlations and PRCC with within-outcome FDR correction.
- 95% confidence intervals in targeted and hysteresis panels.

Upload `msc_model_feedback.py` and `config_msc_feedback.yaml` when prompted, then run all cells.

In [ ]:
# Local, Drive-free reproducible setup
from pathlib import Path
import os, sys, json, hashlib, shutil, warnings, zipfile
from datetime import datetime
from google.colab import files
warnings.filterwarnings('ignore')
ROOT=Path('/content/msc_supplementary_figureS7/output')
MODEL_DIR=ROOT/'model'; DATA=ROOT/'data'; STATISTICS=ROOT/'statistics'; CHECKPOINTS=ROOT/'checkpoints'; MANIFESTS=ROOT/'manifests'; AUDIT_FIGURES=ROOT/'figures_audit'; LOGS=ROOT/'logs'; R_INPUT=ROOT/'10_r_input'
for p in [ROOT,MODEL_DIR,DATA,STATISTICS,CHECKPOINTS,MANIFESTS,AUDIT_FIGURES,LOGS,R_INPUT]:p.mkdir(parents=True,exist_ok=True)
OUTDIR=ROOT; PROJECT_DIR=MODEL_DIR; RUN_DIR=ROOT; RUN_NAME='MSC_SupplementaryFigureS7_reproducible'; IN_COLAB=True
def sha256_file(path,chunk_size=1024*1024):
 d=hashlib.sha256()
 with open(path,'rb') as h:
  for c in iter(lambda:h.read(chunk_size),b''):d.update(c)
 return d.hexdigest()
def place(source,destination):
 source,destination=Path(source),Path(destination)
 if source.resolve()!=destination.resolve():shutil.copy2(source,destination)
MODEL_UPLOAD=Path('/content/msc_model_feedback.py'); CONFIG_UPLOAD=Path('/content/config_msc_feedback.yaml')
if not (MODEL_UPLOAD.exists() and CONFIG_UPLOAD.exists()):
 print('Upload msc_model_feedback.py and config_msc_feedback.yaml now.')
 uploaded=files.upload()
 for name in uploaded:
  path=Path('/content')/name
  if name.endswith('.py'):place(path,MODEL_UPLOAD)
  elif name.endswith(('.yaml','.yml')):place(path,CONFIG_UPLOAD)
if not MODEL_UPLOAD.exists() or not CONFIG_UPLOAD.exists():raise FileNotFoundError('Both model and config are required.')
place(MODEL_UPLOAD,MODEL_DIR/'msc_model_feedback.py'); place(CONFIG_UPLOAD,MODEL_DIR/'config_msc_feedback.yaml')
os.chdir(MODEL_DIR)
if str(MODEL_DIR) not in sys.path:sys.path.insert(0,str(MODEL_DIR))
print('Output root:',ROOT);print('Model hash:',sha256_file(MODEL_DIR/'msc_model_feedback.py')[:16]);print('Config hash:',sha256_file(MODEL_DIR/'config_msc_feedback.yaml')[:16])


## Configuration and validation

The notebook mirrors `msc.sample_params`: Python defaults are loaded first and YAML values are then applied as overrides. This preserves fixed defaults such as `cargo_ell=1.0`.

In [ ]:
# ============================================================
# Configuration, scientific imports, and validation
# ============================================================

# Install all packages required by the original S7 analysis.
!pip -q install pyyaml scipy seaborn joblib tqdm pandas numpy matplotlib

import sys
import json
import yaml
import importlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import (
    qmc,
    rankdata,
    pearsonr,
    spearmanr,
)

from joblib import Parallel, delayed


# ------------------------------------------------------------
# Import the frozen MSC model
# ------------------------------------------------------------
# The first setup cell has already copied the model into MODEL_DIR
# and added MODEL_DIR to sys.path.

assert (MODEL_DIR / "msc_model_feedback.py").exists(), (
    f"Frozen MSC model not found: {MODEL_DIR / 'msc_model_feedback.py'}"
)

assert (MODEL_DIR / "config_msc_feedback.yaml").exists(), (
    f"Frozen MSC configuration not found: "
    f"{MODEL_DIR / 'config_msc_feedback.yaml'}"
)

if str(MODEL_DIR) not in sys.path:
    sys.path.insert(0, str(MODEL_DIR))

importlib.invalidate_caches()

import msc_model_feedback as msc

importlib.reload(msc)


# ------------------------------------------------------------
# Frozen production design
# ------------------------------------------------------------

RUN_MODE = "final"

# Use all available CPU cores. Set this to 1 if Colab memory is limited.
N_JOBS = -1

BASE_SEED = 20260806

N_LHS = 500

N_REPS_GLOBAL = 4

N_REPS_PROFILE = 30

N_PROFILE_POINTS = 7

N_REPS_HYSTERESIS = 15


# ------------------------------------------------------------
# Load configuration
# ------------------------------------------------------------

CONFIG_PATH = MODEL_DIR / "config_msc_feedback.yaml"

with open(
    CONFIG_PATH,
    "r",
    encoding="utf-8",
) as handle:

    CONFIG = yaml.safe_load(handle) or {}


# Begin with the defaults defined by the frozen model, then apply
# the YAML overrides exactly as sample_params() does.

RANGES = dict(msc.DEFAULT_RANGES)

for key, value in (
    CONFIG.get("parameter_ranges") or {}
).items():

    RANGES[key] = tuple(value)


FIXED = dict(msc.DEFAULT_FIXED)

FIXED.update(
    CONFIG.get("fixed") or {}
)


# ------------------------------------------------------------
# Validate the 27 frozen parameter intervals
# ------------------------------------------------------------

EXPECTED = {
    "L": (80, 180),
    "Nsyn": (18, 35),
    "granule_density": (0.60, 1.30),
    "v": (0.05, 0.30),
    "D": (0.005, 0.15),
    "p_switch": (0.002, 0.04),
    "epsilon": (0.20, 0.80),
    "h0": (0.0005, 0.0025),
    "beta_T": (2, 3.8),
    "beta_M": (1, 3),
    "ell_h": (1, 2.2),
    "ell_q_ratio": (0.35, 0.75),
    "alphaS": (0.025, 0.080),
    "q_min": (0.10, 0.25),
    "tau_T": (900, 3600),
    "tau_S": (1200, 7200),
    "delta_translation": (300, 900),
    "M_bas": (0.02, 0.12),
    "M_max": (0.85, 1),
    "k_M": (0.00015, 0.0012),
    "alphaM": (0.0015, 0.010),
    "K_M": (0.40, 0.65),
    "n_hill": (3, 6),
    "strong_fraction": (0.35, 0.60),
    "strong_sigma": (0.18, 0.42),
    "weak_shift": (1.2, 2.4),
    "weak_sigma": (0.65, 1.10),
}


assert len(RANGES) == 27, (
    f"Expected 27 sampled parameter ranges; found {len(RANGES)}."
)


for key, expected_range in EXPECTED.items():

    assert key in RANGES, (
        f"Missing sampled parameter: {key}"
    )

    assert np.allclose(
        RANGES[key],
        expected_range,
        rtol=0,
        atol=1e-12,
    ), (
        key,
        RANGES.get(key),
        expected_range,
    )


# ------------------------------------------------------------
# Validate required fixed settings
# ------------------------------------------------------------

required_fixed = [
    "cargo_ell",
    "D_material",
    "dt",
    "T_total",
]

for key in required_fixed:

    assert key in FIXED, (
        f"Missing fixed parameter: {key}"
    )


assert float(FIXED["cargo_ell"]) == 1.0

assert float(FIXED["D_material"]) == 4.0

assert float(FIXED["dt"]) == 2.0

assert float(FIXED["T_total"]) == 3600.0


# ------------------------------------------------------------
# Confirm model identity and design
# ------------------------------------------------------------

assert "full_msc_feedback" in msc.MODEL_VARIANTS

assert RUN_MODE == "final"

assert N_LHS == 500

assert N_REPS_GLOBAL == 4

assert N_REPS_PROFILE == 30

assert N_PROFILE_POINTS == 7

assert N_REPS_HYSTERESIS == 15


print("Scientific imports loaded.")

print(
    "MSC model:",
    MODEL_DIR / "msc_model_feedback.py",
)

print(
    "Configuration:",
    CONFIG_PATH,
)

print(
    "Model hash:",
    sha256_file(
        MODEL_DIR / "msc_model_feedback.py"
    )[:16],
)

print(
    "Config hash:",
    sha256_file(
        CONFIG_PATH
    )[:16],
)

print(
    "Validated 27 sampled intervals and fixed defaults."
)

print(
    "Production design:",
    f"{N_LHS} LHS sets × {N_REPS_GLOBAL} replicates;",
    f"{N_REPS_PROFILE} targeted-profile replicates;",
    f"{N_REPS_HYSTERESIS} hysteresis replicates per n_H.",
)


display(
    pd.DataFrame(
        RANGES,
        index=[
            "lower",
            "upper",
        ],
    ).T
)

In [ ]:
def sha256(path):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for b in iter(lambda:f.read(1<<20),b""): h.update(b)
    return h.hexdigest()
metadata={"analysis":"MSC Supplementary Figure S7 final","created":datetime.now().isoformat(),
          "run_mode":RUN_MODE,"base_seed":BASE_SEED,"N_LHS":N_LHS,
          "N_REPS_GLOBAL":N_REPS_GLOBAL,"N_REPS_PROFILE":N_REPS_PROFILE,
          "N_PROFILE_POINTS":N_PROFILE_POINTS,"N_REPS_HYSTERESIS":N_REPS_HYSTERESIS,
          "model_sha256":sha256("msc_model_feedback.py"),"config_sha256":sha256("config_msc_feedback.yaml"),
          "parameter_ranges":RANGES,"fixed":FIXED}
with open(OUTDIR/"FigureS7_FINAL_run_metadata.json","w") as f: json.dump(metadata,f,indent=2)

## Shared functions

In [ ]:
INTEGER_PARAMS={"Nsyn","n_hill"}
PARAM_NAMES=list(RANGES)
OUTCOMES=["continuous_specificity","material_memory_specificity","active_strong_enrichment",
          "active_productive_enrichment","control_max_S_late","control_max_M_late",
          "active_S_late","active_M_late","n_docks","n_productive_docks"]
FIG_OUTCOMES=["continuous_specificity","material_memory_specificity","active_strong_enrichment",
              "active_productive_enrichment","control_max_S_late","active_S_late"]
CORE_FREE=["epsilon","h0","beta_T","beta_M","ell_h","ell_q_ratio","alphaS","q_min","k_M","alphaM","K_M","n_hill"]
PRETTY_PARAM={"epsilon":r"$\epsilon$","h0":r"$h_0$","beta_T":r"$\beta_T$","beta_M":r"$\beta_M$",
 "ell_h":r"$\ell_h$","ell_q_ratio":r"$\ell_q/\ell_h$","alphaS":r"$\alpha_S$","q_min":r"$q_{\min}$",
 "k_M":r"$k_M$","alphaM":r"$\alpha_M$","K_M":r"$K_M$","n_hill":r"$n_H$","tau_T":r"$\tau_T$",
 "tau_S":r"$\tau_S$","M_bas":r"$M_{\mathrm{bas}}$","M_max":r"$M_{\max}$","p_switch":r"$p_{\mathrm{switch}}$",
 "strong_fraction":"compatible fraction","strong_sigma":r"$\sigma_{\mathrm{strong}}$",
 "weak_shift":"incompatible shift","weak_sigma":r"$\sigma_{\mathrm{weak}}$","granule_density":"granule density",
 "Nsyn":r"$N_{\mathrm{syn}}$","L":r"$L$","v":r"$v$","D":r"$D$","delta_translation":r"$\Delta_{\mathrm{translation}}$"}
PRETTY_OUT={"continuous_specificity":"consolidation specificity","material_memory_specificity":"material-memory specificity",
 "active_strong_enrichment":"compatible-class enrichment","active_productive_enrichment":"productive-payload enrichment",
 "control_max_S_late":"maximum control S","control_max_M_late":"maximum control M","active_S_late":"active-synapse S"}

def finalize(p):
    p=dict(p); p["Nsyn"]=int(p["Nsyn"]); p["n_hill"]=int(p["n_hill"])
    p["Ngran"]=int(max(10,round(float(p["L"])*float(p["granule_density"]))))
    p["ell_q"]=max(.05,float(p["ell_h"])*float(p["ell_q_ratio"]))
    return p

def midpoint():
    p=dict(FIXED)
    for k,(lo,hi) in RANGES.items():
        lo,hi=float(lo),float(hi)
        p[k]=int(np.floor((lo+hi+1)/2)) if k in INTEGER_PARAMS else (lo+hi)/2
    return finalize(p)

def lhs_table(n,seed):
    U=qmc.LatinHypercube(d=len(PARAM_NAMES),seed=seed).random(n); rows=[]
    for i in range(n):
        p=dict(FIXED)
        for j,k in enumerate(PARAM_NAMES):
            lo,hi=map(float,RANGES[k])
            if k in INTEGER_PARAMS:
                z=int(np.floor(lo+U[i,j]*(hi-lo+1))); z=int(np.clip(z,int(lo),int(hi)))
            else: z=lo+U[i,j]*(hi-lo)
            p[k]=z
        p=finalize(p); p["parameter_set"]=i; rows.append(p)
    return pd.DataFrame(rows)

def run_one(p,seed,parameter_set=None,profile_param=None,profile_value=None):
    drop={"parameter_set","profile_param","profile_value"}
    mp={k:v for k,v in dict(p).items() if k not in drop}
    out=msc.simulate_one(mp,seed=int(seed),variant_name="full_msc_feedback",record_obs=False,record_trace=False)
    row=dict(out["metrics"]); row["sim_seed"]=int(seed)
    if parameter_set is not None: row["parameter_set"]=int(parameter_set)
    if profile_param is not None: row["profile_param"]=profile_param
    if profile_value is not None: row["profile_value"]=profile_value
    return row

def bh(p):
    p=np.asarray(p,float); order=np.argsort(p); r=p[order]
    q=np.minimum.accumulate((r*len(p)/np.arange(1,len(p)+1))[::-1])[::-1]
    out=np.empty_like(q); out[order]=np.clip(q,0,1); return out

def sensitivity_table(df):
    rows=[]
    for out in [o for o in OUTCOMES if o in df]:
        w=df[PARAM_NAMES+[out]].replace([np.inf,-np.inf],np.nan).dropna()
        if len(w)<=len(PARAM_NAMES)+2 or w[out].nunique()<3: continue
        X=np.column_stack([rankdata(w[k].astype(float)) for k in PARAM_NAMES]); y=rankdata(w[out].astype(float))
        for j,pnm in enumerate(PARAM_NAMES):
            other=[k for k in range(len(PARAM_NAMES)) if k!=j]
            Z=np.column_stack([np.ones(len(w)),X[:,other]])
            rx=X[:,j]-Z@np.linalg.lstsq(Z,X[:,j],rcond=None)[0]
            ry=y-Z@np.linalg.lstsq(Z,y,rcond=None)[0]
            r,pv=pearsonr(rx,ry); sr,sp=spearmanr(w[pnm],w[out])
            rows.append({"parameter":pnm,"outcome":out,"prcc":r,"prcc_p":pv,"spearman_rho":sr,"spearman_p":sp,"n":len(w)})
    z=pd.DataFrame(rows); z["prcc_q"]=np.nan; z["spearman_q"]=np.nan
    for out,idx in z.groupby("outcome").groups.items():
        idx=list(idx); z.loc[idx,"prcc_q"]=bh(z.loc[idx,"prcc_p"]); z.loc[idx,"spearman_q"]=bh(z.loc[idx,"spearman_p"])
    return z
REFERENCE=midpoint()
assert all(k in REFERENCE for k in ["cargo_ell","D_material","Ngran","ell_q"])
smoke=dict(REFERENCE); smoke["T_total"]=20.; smoke["activation_time"]=4.
msc.simulate_one(smoke,seed=12345,variant_name="full_msc_feedback")
print("Smoke test passed; reference Ngran =",REFERENCE["Ngran"])

## Global 27-parameter analysis

In [ ]:
# Global sensitivity: cache-safe reuse of complete raw output.
lhs_path=DATA/'FigureS7_FINAL_LHS_parameter_sets.csv'; raw_path=DATA/'FigureS7_FINAL_global_raw.csv'
if lhs_path.exists() and raw_path.exists():
 lhs=pd.read_csv(lhs_path);global_raw=pd.read_csv(raw_path);print('Reusing global block:',global_raw.shape)
else:
 lhs=lhs_table(N_LHS,BASE_SEED);lhs.to_csv(lhs_path,index=False);jobs=[]
 for _,row in lhs.iterrows():
  p=row.to_dict();i=int(row.parameter_set)
  for rep in range(N_REPS_GLOBAL):jobs.append((p,BASE_SEED+100000+i*100+rep,i))
 global_raw=pd.DataFrame(Parallel(n_jobs=N_JOBS,verbose=10)(delayed(run_one)(p,seed,parameter_set=i) for p,seed,i in jobs));global_raw.to_csv(raw_path,index=False)
available=[o for o in OUTCOMES if o in global_raw]
global_mean=lhs.merge(global_raw.groupby('parameter_set',as_index=False)[available].mean(),on='parameter_set');global_mean.to_csv(DATA/'FigureS7_FINAL_global_means.csv',index=False)
sens=sensitivity_table(global_mean);sens.to_csv(STATISTICS/'FigureS7_FINAL_PRCC_Spearman.csv',index=False);display(sens.sort_values('prcc',key=lambda x:x.abs(),ascending=False).head(30))


## Targeted profiles

In [ ]:
# Targeted profiles: cache-safe reuse, summaries always rebuilt.
LOG_PARAMS={'h0','k_M','alphaM'}
def grid(k):
 lo,hi=map(float,RANGES[k])
 if k in INTEGER_PARAMS:return np.arange(int(lo),int(hi)+1)
 return np.geomspace(lo,hi,N_PROFILE_POINTS) if k in LOG_PARAMS else np.linspace(lo,hi,N_PROFILE_POINTS)
raw_path=DATA/'FigureS7_FINAL_profiles_raw.csv'
if raw_path.exists():profile_raw=pd.read_csv(raw_path);print('Reusing profile block:',profile_raw.shape)
else:
 jobs=[]
 for pi,k in enumerate(CORE_FREE):
  for value in grid(k):
   p=finalize({**REFERENCE,k:(int(value) if k in INTEGER_PARAMS else float(value))})
   for rep in range(N_REPS_PROFILE):jobs.append((p,BASE_SEED+300000+pi*10000+rep,k,value))
 profile_raw=pd.DataFrame(Parallel(n_jobs=N_JOBS,verbose=10)(delayed(run_one)(p,seed,profile_param=k,profile_value=v) for p,seed,k,v in jobs));profile_raw.to_csv(raw_path,index=False)
agg={}
for o in [x for x in OUTCOMES if x in profile_raw]:agg[o+'_mean']=(o,'mean');agg[o+'_sem']=(o,lambda x:x.std(ddof=1)/np.sqrt(len(x)))
profile=profile_raw.groupby(['profile_param','profile_value']).agg(**agg).reset_index()
for o in [x for x in OUTCOMES if x in profile_raw]:profile[o+'_ci95']=1.96*profile[o+'_sem']
profile.to_csv(STATISTICS/'FigureS7_FINAL_profiles_summary.csv',index=False)


## Hill-dependent hysteresis

The complete final \(S\) and \(M\) vectors are transferred between successive amplitudes. The Figure 4 protocol uses the explicit duration override `T_total=4200 s`.

In [ ]:
# Stateful Hill hysteresis: cache-safe reuse; the 4200-s protocol is intentional.
HILLS=list(range(int(RANGES['n_hill'][0]),int(RANGES['n_hill'][1])+1));AMPS=np.linspace(0,1.6,9)
def run_hyst(nh,rep):
 p=finalize({**REFERENCE,'n_hill':int(nh),'T_total':4200.});active=[p['Nsyn']//2];iniS=None;iniM=None;rows=[]
 for step,(branch,amp) in enumerate([('up',x) for x in AMPS]+[('down',x) for x in AMPS[::-1]]):
  seed=BASE_SEED+700000+nh*10000+rep*100+step;out=msc.simulate_one(p,seed=seed,variant_name='full_msc_feedback',active_synapses=active,activation_amp=float(amp),initial_S=iniS,initial_M=iniM,record_trace=True)
  Sh,Mh=out['S_history'],out['M_history'];iniS={i:float(Sh[-1,i]) for i in range(p['Nsyn'])};iniM={i:float(Mh[-1,i]) for i in range(p['Nsyn'])};rows.append({'n_hill':nh,'rep':rep,'branch':branch,'amp':amp,'S':out['metrics']['active_S_late'],'M':out['metrics']['active_M_late'],'sim_seed':seed})
 return rows
raw_path=DATA/'FigureS7_FINAL_hysteresis_raw.csv'
if raw_path.exists():hraw=pd.read_csv(raw_path);print('Reusing hysteresis block:',hraw.shape)
else:
 nested=Parallel(n_jobs=N_JOBS,verbose=10)(delayed(run_hyst)(n,r) for n in HILLS for r in range(N_REPS_HYSTERESIS));hraw=pd.DataFrame([x for block in nested for x in block]);hraw.to_csv(raw_path,index=False)
gaps=[]
for (n,r),d in hraw.groupby(['n_hill','rep']):
 u=d[d.branch=='up'][['amp','S','M']].rename(columns={'S':'S_up','M':'M_up'});v=d[d.branch=='down'][['amp','S','M']].rename(columns={'S':'S_down','M':'M_down'});g=u.merge(v,on='amp').sort_values('amp');g['S_gap']=g.S_down-g.S_up;g['M_gap']=g.M_down-g.M_up;gaps.append({'n_hill':n,'rep':r,'S_auc_gap':np.trapz(g.S_gap,g.amp),'M_auc_gap':np.trapz(g.M_gap,g.amp),'max_S_gap':g.S_gap.max(),'max_M_gap':g.M_gap.max()})
hg=pd.DataFrame(gaps);hg.to_csv(DATA/'FigureS7_FINAL_hysteresis_gaps.csv',index=False)
sem=lambda x:x.std(ddof=1)/np.sqrt(len(x))
hsum=hg.groupby('n_hill').agg(n=('rep','size'),S_auc_mean=('S_auc_gap','mean'),S_auc_sem=('S_auc_gap',sem),M_auc_mean=('M_auc_gap','mean'),M_auc_sem=('M_auc_gap',sem),max_S_gap_mean=('max_S_gap','mean'),max_M_gap_mean=('max_M_gap','mean')).reset_index();hsum['S_auc_ci95']=1.96*hsum.S_auc_sem;hsum['M_auc_ci95']=1.96*hsum.M_auc_sem;hsum.to_csv(STATISTICS/'FigureS7_FINAL_hysteresis_summary.csv',index=False);display(hsum)


## Publication figure

Panel A marks FDR-adjusted PRCC values with an asterisk when \(q<0.05\). Panel B ranks parameters by the mean, rather than maximum, absolute PRCC across the displayed outcomes. Panel D is normalized separately within each outcome column and is explicitly labelled as such. Panel E uses separate y-axes for state variables and enrichment.

In [ ]:
sns.set_theme(style="whitegrid",context="paper",font_scale=1.05)
fig,ax=plt.subplots(2,3,figsize=(18,11.5),constrained_layout=True)
# A
sub=sens[sens.parameter.isin(CORE_FREE)&sens.outcome.isin(FIG_OUTCOMES)].copy()
mat=sub.pivot(index="parameter",columns="outcome",values="prcc").reindex(CORE_FREE)
qmat=sub.pivot(index="parameter",columns="outcome",values="prcc_q").reindex(CORE_FREE)
annot=mat.copy().astype(object)
for i in range(mat.shape[0]):
    for j in range(mat.shape[1]):
        x=mat.iloc[i,j]; q=qmat.iloc[i,j]
        annot.iloc[i,j]="" if pd.isna(x) else f"{x:.2f}{'*' if pd.notna(q) and q<.05 else ''}"
mat.index=[PRETTY_PARAM.get(x,x) for x in mat.index]; mat.columns=[PRETTY_OUT.get(x,x) for x in mat.columns]
annot.index=mat.index; annot.columns=mat.columns
sns.heatmap(mat,cmap="vlag",center=0,vmin=-1,vmax=1,annot=annot,fmt="",annot_kws={"fontsize":7},ax=ax[0,0],cbar_kws={"label":"PRCC"})
ax[0,0].set_title("A  Partial-rank sensitivity (* FDR q < 0.05)"); ax[0,0].set_xlabel(""); ax[0,0].set_ylabel("")
ax[0,0].tick_params(axis="x",rotation=55)
# B
ranked=(sens[sens.outcome.isin(FIG_OUTCOMES)].assign(a=lambda d:d.prcc.abs()).groupby("parameter").a.mean().sort_values().tail(15))
ax[0,1].barh([PRETTY_PARAM.get(x,x) for x in ranked.index],ranked.values,color="#3973ac")
ax[0,1].set_xlabel("Mean |PRCC| across displayed outcomes"); ax[0,1].set_title("B  Global influence ranking")
# C
cs=sens[sens.outcome=="continuous_specificity"].copy(); cs=cs.iloc[np.argsort(abs(cs.prcc.values))[-15:]].sort_values("prcc")
ax[0,2].barh([PRETTY_PARAM.get(x,x) for x in cs.parameter],cs.prcc,color=np.where(cs.prcc>=0,"#2a9d8f","#e76f51"))
for y,(_,r) in enumerate(cs.reset_index(drop=True).iterrows()):
    if r.prcc_q<.05: ax[0,2].text(r.prcc+(0.015 if r.prcc>=0 else -0.015),y,"*",va="center",ha="left" if r.prcc>=0 else "right")
ax[0,2].axvline(0,color="black",lw=.8); ax[0,2].set_xlabel("PRCC"); ax[0,2].set_title("C  Continuous consolidation specificity")
# D
outs=["continuous_specificity","material_memory_specificity","active_productive_enrichment","control_max_S_late"]
erows=[]
for k in CORE_FREE:
    d=profile[profile.profile_param==k].sort_values("profile_value"); row={"parameter":k}
    for o in outs: row[o]=d[o+"_mean"].iloc[-1]-d[o+"_mean"].iloc[0]
    erows.append(row)
eff=pd.DataFrame(erows).set_index("parameter"); eff.to_csv(OUTDIR/"FigureS7_FINAL_profile_endpoint_effects.csv")
norm=eff.copy()
for c in norm: norm[c]/=max(np.nanmax(abs(norm[c])),1e-12)
norm.index=[PRETTY_PARAM.get(x,x) for x in norm.index]; norm.columns=[PRETTY_OUT.get(x,x) for x in norm.columns]
sns.heatmap(norm,cmap="vlag",center=0,vmin=-1,vmax=1,ax=ax[1,0],cbar_kws={"label":"Column-normalized high-low effect"})
ax[1,0].set_title("D  Targeted profiles (normalized within outcome)"); ax[1,0].set_xlabel(""); ax[1,0].set_ylabel(""); ax[1,0].tick_params(axis="x",rotation=55)
# E
nh=profile[profile.profile_param=="n_hill"].sort_values("profile_value")
left=ax[1,1]
left.errorbar(nh.profile_value,nh.active_S_late_mean,yerr=nh.active_S_late_ci95,marker="o",capsize=3,label="late S",color="#264653")
left.errorbar(nh.profile_value,nh.active_M_late_mean,yerr=nh.active_M_late_ci95,marker="o",capsize=3,label="late M",color="#e76f51")
right=left.twinx(); right.errorbar(nh.profile_value,nh.active_productive_enrichment_mean,yerr=nh.active_productive_enrichment_ci95,
 marker="s",capsize=3,label="productive enrichment",color="#2a9d8f")
left.set_xticks(HILLS); left.set_xlabel(r"Hill coefficient $n_H$"); left.set_ylabel("State value (95% CI)"); right.set_ylabel("Enrichment ratio (95% CI)")
lines=left.lines+right.lines; left.legend(lines,[x.get_label() for x in lines],frameon=False,loc="best"); left.set_title("E  Hill-coefficient profile")
# F
ax[1,2].errorbar(hsum.n_hill,hsum.S_auc_mean,yerr=hsum.S_auc_ci95,marker="o",capsize=3,label="S",color="#264653")
ax[1,2].errorbar(hsum.n_hill,hsum.M_auc_mean,yerr=hsum.M_auc_ci95,marker="o",capsize=3,label="M",color="#e76f51")
ax[1,2].axhline(0,color="black",lw=.8); ax[1,2].set_xticks(HILLS); ax[1,2].set_xlabel(r"Hill coefficient $n_H$")
ax[1,2].set_ylabel("Integrated branch separation (95% CI)"); ax[1,2].set_title("F  Hill-dependent hysteresis"); ax[1,2].legend(frameon=False)
fig.suptitle("MSC Supplementary Figure S7 — global and targeted sensitivity analysis",fontsize=16)
fig.text(.5,-.015,f"Global: {N_LHS} parameter sets x {N_REPS_GLOBAL} stochastic replicates; profiles: {N_REPS_PROFILE} replicates per point; hysteresis: {N_REPS_HYSTERESIS} replicates per n_H",ha="center",fontsize=9)
fig.savefig(OUTDIR/"Supplementary_Figure_S7_FINAL.png",dpi=300,bbox_inches="tight")
fig.savefig(OUTDIR/"Supplementary_Figure_S7_FINAL.pdf",bbox_inches="tight")
plt.show()

## Quality control and download

In [ ]:
qc={'run_mode':RUN_MODE,'n_parameter_sets':int(lhs.parameter_set.nunique()),'global_replicates_per_set':int(global_raw.groupby('parameter_set').size().min()),'n_global_simulations':len(global_raw),'profile_parameters':sorted(profile_raw.profile_param.unique()),'profile_min_replicates':int(profile_raw.groupby(['profile_param','profile_value']).size().min()),'hill_levels':HILLS,'hysteresis_replicates_per_level':int(hg.groupby('n_hill').size().min()),'n_sensitivity_rows':len(sens)}
assert qc['n_parameter_sets']==500 and qc['global_replicates_per_set']==4 and qc['profile_min_replicates']==30 and qc['hysteresis_replicates_per_level']==15
json.dump(qc,open(STATISTICS/'FigureS7_FINAL_quality_control.json','w'),indent=2);display(pd.Series(qc))


## Interpretation boundaries

PRCC estimates monotonic parameter influence conditional on the other sampled parameters; it does not establish biological causality. A weak PRCC can reflect weak influence, non-monotonicity, interactions, saturation, or stochastic noise. Sensitivity of output magnitude is not itself failure of robustness. The central test is whether material-dependent enrichment, low crosstalk, and feedback-dependent history dependence persist across the prespecified intervals. The Hill function does not itself prove bistability; the hysteresis protocol tests history dependence in the implemented stochastic system. All MSC findings remain computational predictions requiring experimental validation.

## Canonical exports and reproducibility bundle

In [ ]:
# Canonical exports, manifest and one downloadable archive.
required=[DATA/'FigureS7_FINAL_LHS_parameter_sets.csv',DATA/'FigureS7_FINAL_global_raw.csv',DATA/'FigureS7_FINAL_global_means.csv',STATISTICS/'FigureS7_FINAL_PRCC_Spearman.csv',DATA/'FigureS7_FINAL_profiles_raw.csv',STATISTICS/'FigureS7_FINAL_profiles_summary.csv',DATA/'FigureS7_FINAL_hysteresis_raw.csv',DATA/'FigureS7_FINAL_hysteresis_gaps.csv',STATISTICS/'FigureS7_FINAL_hysteresis_summary.csv',STATISTICS/'FigureS7_FINAL_quality_control.json']
missing=[str(p) for p in required if not p.exists() or p.stat().st_size<=0]
if missing:raise RuntimeError('Required outputs missing:\n'+'\n'.join(missing))
assert len(pd.read_csv(DATA/'FigureS7_FINAL_global_raw.csv'))==N_LHS*N_REPS_GLOBAL
assert pd.read_csv(DATA/'FigureS7_FINAL_profiles_raw.csv').groupby(['profile_param','profile_value']).size().ge(N_REPS_PROFILE).all()
assert pd.read_csv(DATA/'FigureS7_FINAL_hysteresis_raw.csv').groupby(['n_hill','rep']).size().eq(2*len(AMPS)).all()
for p in required:shutil.copy2(p,R_INPUT/p.name)
aliases={'panel_A_PRCC_heatmap.csv':STATISTICS/'FigureS7_FINAL_PRCC_Spearman.csv','panel_B_global_influence_ranking.csv':STATISTICS/'FigureS7_FINAL_PRCC_Spearman.csv','panel_C_consolidation_PRCC.csv':STATISTICS/'FigureS7_FINAL_PRCC_Spearman.csv','panel_D_targeted_profiles.csv':STATISTICS/'FigureS7_FINAL_profiles_summary.csv','panel_E_hill_profiles.csv':DATA/'FigureS7_FINAL_hysteresis_raw.csv','panel_F_hysteresis.csv':STATISTICS/'FigureS7_FINAL_hysteresis_summary.csv'}
for name,p in aliases.items():shutil.copy2(p,R_INPUT/name)
metadata={'analysis':'MSC Supplementary Figure S7 sensitivity','version':'local_reproducibility_v1','run_mode':RUN_MODE,'base_seed':BASE_SEED,'design':{'N_LHS':N_LHS,'N_REPS_GLOBAL':N_REPS_GLOBAL,'N_REPS_PROFILE':N_REPS_PROFILE,'N_PROFILE_POINTS':N_PROFILE_POINTS,'N_REPS_HYSTERESIS':N_REPS_HYSTERESIS,'hill_levels':[int(x) for x in HILLS],'hysteresis_amplitudes':[float(x) for x in AMPS],'profile_parameters':CORE_FREE,'PRCC_FDR':'BH within outcome'},'validation':qc,'model_sha256':sha256_file(MODEL_DIR/'msc_model_feedback.py'),'config_sha256':sha256_file(MODEL_DIR/'config_msc_feedback.yaml'),'generated_at':datetime.utcnow().isoformat()+'Z','outputs':{str(p.relative_to(ROOT)):sha256_file(p) for p in sorted(ROOT.rglob('*')) if p.is_file()}}
(ROOT/'MANIFEST_S7.json').write_text(json.dumps(metadata,indent=2))
(ROOT/'README.md').write_text('''# MSC Supplementary Figure S7 reproducibility bundle

Global analysis: 500 Latin-hypercube sets across 27 sampled parameters with four stochastic replicates each. Targeted profiles use 30 replicates per point. Stateful hysteresis uses n_H=3,4,5,6, 15 replicates per level, and the 4200-s stepped-drive protocol.

10_r_input contains canonical R-renderer inputs. All values are stochastic in-silico predictions, not wet-lab measurements.
''')
zip_path=Path('/content/MSC_SupplementaryFigureS7_sensitivity_bundle.zip')
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
 for p in sorted(ROOT.rglob('*')):
  if p.is_file():z.write(p,p.relative_to(ROOT.parent))
assert zip_path.exists() and zip_path.stat().st_size>0
print('VALIDATION PASSED');print('ZIP:',zip_path.name,f'{zip_path.stat().st_size/1e6:.1f} MB');print('R inputs:',len(list(R_INPUT.iterdir())));files.download(str(zip_path))
